In [1]:
from typing import TypedDict, List
from urllib import response

from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage
from langgraph.graph import StateGraph, START, END


In [3]:
class AgentState(TypedDict):
    messages: List[HumanMessage]

llm = ChatOllama(model="llama3.2:3b")

def process(state: AgentState) -> AgentState:
    response = llm.invoke(state["messages"])
    print(f"\nAI: {response.content}")
    return state

graph = StateGraph(AgentState)

graph.add_node("process",process)

graph.add_edge(START, "process")
graph.add_edge("process",END)

agent = graph.compile()

user_input = input("Input :")
while user_input != "exit":
    result = agent.invoke({"messages":[HumanMessage(content=user_input)]})
    user_input = input("Input :")


AI: I'm an artificial intelligence model known as Llama. Llama stands for "Large Language Model Meta AI."


{'messages': [HumanMessage(content='who are you', additional_kwargs={}, response_metadata={})]}

In [4]:
# AI Agent 2 ==> WE ADD MEMORY
from typing import TypedDict, List, Union
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, AIMessage
from langgraph.graph import StateGraph, START, END

class AgentState(TypedDict):
    messages: List[Union[HumanMessage, AIMessage]] # It allows to store Human and AI messages - Datatypes Custom

llm = ChatOllama(model="llama3.2:3b")

def process(state: AgentState) -> AgentState:
    """This node will solve the request you input"""
    output = llm.invoke(state["messages"])
    state["messages"].append(AIMessage(content=output.content))
    print(f"\nAI: {output.content}")
    return state

graph = StateGraph(AgentState)

graph.add_node("process",process)

graph.add_edge(START, "process")
graph.add_edge("process",END)

agent = graph.compile()

conversation_history = []

user_input = input("Input :")
while user_input != "exit":
    conversation_history.append(HumanMessage(content=user_input))
    result = agent.invoke({"messages":conversation_history})
    # print(result["messages"])
    conversation_history = result["messages"]
    user_input = input("Input :")

KeyboardInterrupt: Interrupted by user

In [ ]:
# Agent 3
# REACT AGENT

In [6]:
from typing import TypedDict, List, Union, Annotated, Sequence
from langchain_ollama import ChatOllama
from langchain_core.messages import  BaseMessage
from langchain_core.messages import ToolMessage # Result from tool call
from langchain_core.messages import SystemMessage # System Instructions
from langchain_core.messages import HumanMessage, AIMessage
from langgraph.graph import StateGraph, START, END
from langchain_core.tools import tool
from langgraph.prebuilt import ToolNode
from langgraph.graph.message import add_messages

# Annotated - provides additional context

# email = Annotated[str,"This should be a valid email format"]
# print(email.__metadata__)

#Sequence ==> Auto handle list manipulations

# BaseMessage - parent class of all other messages - AI message, Human Message ...

# add messages ==> reducer function - how to merge ==> Auto mate append

graph = StateGraph(AgentState)

class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], add_messages]

@tool
def add(a: int, b: int) :
    """addition function that adds two numbers"""
    return a + b

tools = [add]

llm = ChatOllama(model="llama3.2:3b").bind_tools(tools)

def model_call(state: AgentState) -> AgentState:
    response = llm.invoke(["You are my AI assistant, answer my query to the best of your ability"])
    return {"messages":[response]}


('This should be a valid email format',)
